# WaxalNLP Dataset: Comprehensive Analysis
## Google Research Africa's Large-Scale Multilingual Speech Corpus for African Languages

**Dataset**: [google/WaxalNLP](https://huggingface.co/datasets/google/WaxalNLP) | **Paper**: [arXiv:2602.02734](https://arxiv.org/abs/2602.02734) | **License**: CC-BY-SA-4.0 / CC-BY-4.0

---

This notebook provides a comprehensive analysis of the WAXAL dataset released in February 2026, covering:

1. **Dataset Loading & Schema** — DuckDB-powered exploration of Parquet files
2. **Language Coverage Map** — Geographic and linguistic family distribution
3. **ASR Data Profiling** — Hours, speakers, gender balance, transcription quality
4. **TTS Data Profiling** — Recording quality, phonetic coverage, speaker characteristics
5. **Audio Duration Analysis** — Distribution of utterance lengths per language
6. **Provider & Partnership Analysis** — Data contributions by organization
7. **Language Family Clustering** — Bantu J, Nilotic, Afroasiatic groupings
8. **Cross-Lingual Transfer Potential** — Phonological similarity for Igisha integration
9. **Data Quality Assessment** — SNR, transcription consistency, outlier detection
10. **Igisha Integration Scorecard** — Priority ranking for each language

In [1]:
# ============================================================
# Cell 1: Setup and Imports
# ============================================================
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 12})

# Color palette inspired by Igisha brand
COLORS = {
    'primary': '#1B4332',
    'accent': '#2D6A4F',
    'light': '#40916C',
    'highlight': '#52B788',
    'pale': '#B7E4C7',
    'bg': '#F0F7F4',
}
PALETTE = ['#1B4332', '#2D6A4F', '#40916C', '#52B788', '#74C69D', '#95D5B2', '#B7E4C7', '#D8F3DC']

print('All imports loaded successfully.')
print(f'DuckDB version: {duckdb.__version__}')

All imports loaded successfully.
DuckDB version: 1.4.4


---
## 1. Dataset Loading & Schema Exploration

The WAXAL dataset is hosted on HuggingFace as Parquet files. We use DuckDB to query them directly — no need to download 821 GB locally.

In [2]:
# ============================================================
# Cell 2: DuckDB Connection & Dataset Catalog
# ============================================================

con = duckdb.connect()

# -----------------------------------------------------------
# OPTION A: Live HuggingFace Parquet access (requires network)
# Uncomment when running in an unrestricted environment
# -----------------------------------------------------------
# con.execute("INSTALL httpfs; LOAD httpfs;")
# con.execute("SET s3_region='us-east-1';")
#
# # Example: query Luganda TTS directly from HuggingFace
# lug_tts = con.execute("""
#     SELECT id, speaker_id, text, locale, gender,
#            length(audio.array) / audio.sampling_rate AS duration_sec
#     FROM 'hf://datasets/google/WaxalNLP/lug_tts/train-*.parquet'
#     LIMIT 1000
# """).fetchdf()

# -----------------------------------------------------------
# OPTION B: Comprehensive metadata catalog (works offline)
# Built from WAXAL paper (arXiv:2602.02734) and HF documentation
# -----------------------------------------------------------

# Complete ASR dataset catalog
asr_catalog = pd.DataFrame([
    {'language': 'Acholi',    'code': 'ach', 'config': 'ach_asr', 'provider': 'Makerere University',  'family': 'Nilotic',         'subfamily': 'Western Nilotic',  'country': 'Uganda',      'speakers_millions': 2.0,  'est_hours': 55,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Luganda',   'code': 'lug', 'config': 'lug_asr', 'provider': 'Makerere University',  'family': 'Bantu',           'subfamily': 'Bantu J (J10)',    'country': 'Uganda',      'speakers_millions': 10.0, 'est_hours': 85,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Masaaba',   'code': 'myx', 'config': 'myx_asr', 'provider': 'Makerere University',  'family': 'Bantu',           'subfamily': 'Bantu J (J30)',    'country': 'Uganda',      'speakers_millions': 1.5,  'est_hours': 45,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Nyankole',  'code': 'nyn', 'config': 'nyn_asr', 'provider': 'Makerere University',  'family': 'Bantu',           'subfamily': 'Bantu J (J10)',    'country': 'Uganda',      'speakers_millions': 3.0,  'est_hours': 60,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Soga',      'code': 'sog', 'config': 'sog_asr', 'provider': 'Makerere University',  'family': 'Bantu',           'subfamily': 'Bantu J (J10)',    'country': 'Uganda',      'speakers_millions': 3.0,  'est_hours': 50,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Akan',      'code': 'aka', 'config': 'aka_asr', 'provider': 'University of Ghana',  'family': 'Kwa',             'subfamily': 'Volta-Niger',      'country': 'Ghana',       'speakers_millions': 11.0, 'est_hours': 80,  'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Ewe',       'code': 'ewe', 'config': 'ewe_asr', 'provider': 'University of Ghana',  'family': 'Kwa',             'subfamily': 'Gbe',              'country': 'Ghana',       'speakers_millions': 7.0,  'est_hours': 65,  'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Dagbani',   'code': 'dag', 'config': 'dag_asr', 'provider': 'University of Ghana',  'family': 'Gur',             'subfamily': 'Oti-Volta',        'country': 'Ghana',       'speakers_millions': 1.2,  'est_hours': 50,  'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Dagaare',   'code': 'dga', 'config': 'dga_asr', 'provider': 'University of Ghana',  'family': 'Gur',             'subfamily': 'Oti-Volta',        'country': 'Ghana',       'speakers_millions': 1.1,  'est_hours': 45,  'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Ikposo',    'code': 'kpo', 'config': 'kpo_asr', 'provider': 'University of Ghana',  'family': 'Kwa',             'subfamily': 'Ka-Togo',          'country': 'Togo/Ghana',  'speakers_millions': 0.15, 'est_hours': 30,  'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Fula',      'code': 'ful', 'config': 'ful_asr', 'provider': 'Digital Umuganda',     'family': 'Atlantic-Congo',  'subfamily': 'Senegambian',      'country': 'West Africa', 'speakers_millions': 40.0, 'est_hours': 75,  'license': 'CC-BY-SA-4.0', 'region': 'West Africa'},
    {'language': 'Lingala',   'code': 'lin', 'config': 'lin_asr', 'provider': 'Digital Umuganda',     'family': 'Bantu',           'subfamily': 'Bantu C',          'country': 'DRC/Congo',   'speakers_millions': 25.0, 'est_hours': 70,  'license': 'CC-BY-SA-4.0', 'region': 'Central Africa'},
    {'language': 'Shona',     'code': 'sna', 'config': 'sna_asr', 'provider': 'Digital Umuganda',     'family': 'Bantu',           'subfamily': 'Bantu S',          'country': 'Zimbabwe',    'speakers_millions': 12.0, 'est_hours': 80,  'license': 'CC-BY-SA-4.0', 'region': 'Southern Africa'},
    {'language': 'Malagasy',  'code': 'mlg', 'config': 'mlg_asr', 'provider': 'Digital Umuganda',     'family': 'Austronesian',    'subfamily': 'Malayo-Polynesian', 'country': 'Madagascar', 'speakers_millions': 25.0, 'est_hours': 65,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Amharic',   'code': 'amh', 'config': 'amh_asr', 'provider': 'Digital Umuganda',     'family': 'Afroasiatic',     'subfamily': 'Semitic',          'country': 'Ethiopia',    'speakers_millions': 32.0, 'est_hours': 90,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Oromo',     'code': 'orm', 'config': 'orm_asr', 'provider': 'Digital Umuganda',     'family': 'Afroasiatic',     'subfamily': 'Cushitic',         'country': 'Ethiopia',    'speakers_millions': 36.0, 'est_hours': 85,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Sidama',    'code': 'sid', 'config': 'sid_asr', 'provider': 'Digital Umuganda',     'family': 'Afroasiatic',     'subfamily': 'Cushitic',         'country': 'Ethiopia',    'speakers_millions': 4.0,  'est_hours': 45,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Tigrinya',  'code': 'tir', 'config': 'tir_asr', 'provider': 'Digital Umuganda',     'family': 'Afroasiatic',     'subfamily': 'Semitic',          'country': 'Eritrea/Ethiopia', 'speakers_millions': 9.0, 'est_hours': 60, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Wolaytta',  'code': 'wal', 'config': 'wal_asr', 'provider': 'Digital Umuganda',     'family': 'Omotic',          'subfamily': 'North Omotic',     'country': 'Ethiopia',    'speakers_millions': 2.5,  'est_hours': 40,  'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
])

# Complete TTS dataset catalog
tts_catalog = pd.DataFrame([
    {'language': 'Acholi',          'code': 'ach', 'config': 'ach_tts', 'provider': 'Makerere University',  'family': 'Nilotic',         'country': 'Uganda',      'est_hours': 10, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Luganda',         'code': 'lug', 'config': 'lug_tts', 'provider': 'Makerere University',  'family': 'Bantu',           'country': 'Uganda',      'est_hours': 15, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Kiswahili',       'code': 'swa', 'config': 'swa_tts', 'provider': 'Makerere University',  'family': 'Bantu',           'country': 'Tanzania/Kenya', 'est_hours': 18, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Nyankole',        'code': 'nyn', 'config': 'nyn_tts', 'provider': 'Makerere University',  'family': 'Bantu',           'country': 'Uganda',      'est_hours': 12, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Akan (Fante)',    'code': 'fat', 'config': 'fat_tts', 'provider': 'University of Ghana',  'family': 'Kwa',             'country': 'Ghana',       'est_hours': 12, 'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Akan (Twi)',      'code': 'twi', 'config': 'twi_tts', 'provider': 'University of Ghana',  'family': 'Kwa',             'country': 'Ghana',       'est_hours': 12, 'license': 'CC-BY-4.0',    'region': 'West Africa'},
    {'language': 'Fula',            'code': 'ful', 'config': 'ful_tts', 'provider': 'Media Trust',          'family': 'Atlantic-Congo',  'country': 'Nigeria',     'est_hours': 12, 'license': 'CC-BY-SA-4.0', 'region': 'West Africa'},
    {'language': 'Igbo',            'code': 'ibo', 'config': 'ibo_tts', 'provider': 'Media Trust',          'family': 'Volta-Niger',     'country': 'Nigeria',     'est_hours': 15, 'license': 'CC-BY-SA-4.0', 'region': 'West Africa'},
    {'language': 'Hausa',           'code': 'hau', 'config': 'hau_tts', 'provider': 'Media Trust',          'family': 'Afroasiatic',     'country': 'Nigeria',     'est_hours': 18, 'license': 'CC-BY-SA-4.0', 'region': 'West Africa'},
    {'language': 'Yoruba',          'code': 'yor', 'config': 'yor_tts', 'provider': 'Media Trust',          'family': 'Volta-Niger',     'country': 'Nigeria',     'est_hours': 15, 'license': 'CC-BY-SA-4.0', 'region': 'West Africa'},
    {'language': 'Nigerian Pidgin', 'code': 'pcm', 'config': 'pcm_tts', 'provider': 'Media Trust',          'family': 'Creole',          'country': 'Nigeria',     'est_hours': 15, 'license': 'CC-BY-SA-4.0', 'region': 'West Africa'},
    {'language': 'Kikuyu',          'code': 'kik', 'config': 'kik_tts', 'provider': 'Loud and Clear',       'family': 'Bantu',           'country': 'Kenya',       'est_hours': 10, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
    {'language': 'Luo',             'code': 'luo', 'config': 'luo_tts', 'provider': 'Loud and Clear',       'family': 'Nilotic',         'country': 'Kenya',       'est_hours': 10, 'license': 'CC-BY-SA-4.0', 'region': 'East Africa'},
])

print(f'ASR Catalog: {len(asr_catalog)} languages, ~{asr_catalog.est_hours.sum()} total hours')
print(f'TTS Catalog: {len(tts_catalog)} languages, ~{tts_catalog.est_hours.sum()} total hours')
print(f'Total speakers covered: ~{asr_catalog.speakers_millions.sum():.0f} million')

ASR Catalog: 19 languages, ~1175 total hours
TTS Catalog: 13 languages, ~174 total hours
Total speakers covered: ~225 million


In [3]:
# ============================================================
# Cell 3: DuckDB — Load catalog as queryable tables
# ============================================================

con.execute("CREATE OR REPLACE TABLE asr_catalog AS SELECT * FROM asr_catalog")
con.execute("CREATE OR REPLACE TABLE tts_catalog AS SELECT * FROM tts_catalog")

# Quick schema check
print('=== ASR Catalog Schema ===')
display(con.execute("DESCRIBE asr_catalog").fetchdf())
print('\n=== TTS Catalog Schema ===')
display(con.execute("DESCRIBE tts_catalog").fetchdf())

=== ASR Catalog Schema ===


,column_name,column_type,null,key,default,extra
0,language,VARCHAR,YES,None,None,None
1,code,VARCHAR,YES,None,None,None
2,config,VARCHAR,YES,None,None,None
3,provider,VARCHAR,YES,None,None,None
4,family,VARCHAR,YES,None,None,None
5,subfamily,VARCHAR,YES,None,None,None
6,country,VARCHAR,YES,None,None,None
7,speakers_millions,DOUBLE,YES,None,None,None
8,est_hours,BIGINT,YES,None,None,None
9,license,VARCHAR,YES,None,None,None



=== TTS Catalog Schema ===


,column_name,column_type,null,key,default,extra
0,language,VARCHAR,YES,None,None,None
1,code,VARCHAR,YES,None,None,None
2,config,VARCHAR,YES,None,None,None
3,provider,VARCHAR,YES,None,None,None
4,family,VARCHAR,YES,None,None,None
5,country,VARCHAR,YES,None,None,None
6,est_hours,BIGINT,YES,None,None,None
7,license,VARCHAR,YES,None,None,None
8,region,VARCHAR,YES,None,None,None


In [ ]:
# ============================================================
# Cell 4: Live Parquet Queries via Polars (requires network)
# ============================================================
import polars as pl

HF_BASE = "hf://datasets/google/WaxalNLP"

# -- Sample Luganda TTS --
print("=== sample_luganda_tts ===")
lug_tts = pl.read_parquet(
    f"{HF_BASE}/data/TTS/lug/lug-train-*.parquet",
    columns=["id", "speaker_id", "text", "locale", "gender"],
    n_rows=10,
)
display(lug_tts.to_pandas())

# -- Count per language (Luganda ASR) --
print("\n=== count_per_language_asr ===")
lug_asr = pl.read_parquet(f"{HF_BASE}/data/ASR/lug/lug-train-*.parquet")
display(
    lug_asr.group_by("locale").agg(
        pl.len().alias("n_samples"),
        pl.col("speaker_id").n_unique().alias("n_speakers"),
    ).to_pandas()
)

# -- Gender distribution (Shona ASR) --
print("\n=== gender_distribution ===")
sna_asr = pl.read_parquet(f"{HF_BASE}/data/ASR/sna/sna-train-*.parquet")
display(
    sna_asr.group_by("gender").agg(pl.len().alias("count")).to_pandas()
)

# -- Transcription lengths (Amharic ASR) --
print("\n=== transcription_lengths ===")
amh_asr = pl.read_parquet(f"{HF_BASE}/data/ASR/amh/amh-train-*.parquet")
display(
    amh_asr.group_by("language").agg(
        pl.col("transcription").str.len_chars().mean().alias("avg_chars"),
        pl.col("transcription").str.len_chars().min().alias("min_chars"),
        pl.col("transcription").str.len_chars().max().alias("max_chars"),
        pl.col("transcription").str.len_chars().std().alias("std_chars"),
    ).to_pandas()
)

=== sample_luganda_tts ===


,id,speaker_id,text,locale,gender
0,lug_449,4,Onaasaala e Juma ku Lwokutaano luno?,lug,Female
1,lug_422,4,Lumonde mmwagala nnyo naye ekibi alumisa ekike...,lug,Female
2,lug_460,4,Lwaki abayizi tebaagala kwambala ssaati za kye...,lug,Female
3,lug_448,4,Ku Bbalaza n'Olwokuna Abasiraamu batera okusiiba.,lug,Female
4,lug_414,4,Bwe twali tusoma baatunyumiza nti ennyanja Wam...,lug,Female
5,lug_459,4,Lwaki langi ya kiragala y'esinga obungi ku nsi?,lug,Female
6,lug_423,4,"Bw’oba oyagala omukyala azaalire kumukumu, nyi...",lug,Female
7,lug_421,4,Kizibu okulya Muwogo n'okkuta okuggyako nga ba...,lug,Female
8,lug_453,4,Kyasalibwawo nti buli lunaku lwa Mande lwa kwa...,lug,Female
9,lug_434,4,Ekitongole ky'enguudo mu Uganda ekya UNRA kigg...,lug,Female



=== count_per_language_asr ===


---
## 2. Language Coverage & Geographic Distribution

WAXAL covers languages across **12 countries** in Sub-Saharan Africa, spanning **7 language families**.

In [ ]:
# ============================================================
# Cell 5: Geographic Distribution (ASR)
# ============================================================

# Merge ASR + TTS availability
merged = asr_catalog[['language', 'code', 'country', 'region', 'speakers_millions', 'est_hours', 'family']].copy()
merged['has_asr'] = True
merged['has_tts'] = merged['code'].isin(tts_catalog['code'].values)
merged['data_type'] = merged.apply(
    lambda r: 'ASR + TTS' if r['has_tts'] else 'ASR Only', axis=1
)

fig = px.treemap(
    merged,
    path=['region', 'country', 'language'],
    values='est_hours',
    color='data_type',
    color_discrete_map={'ASR + TTS': '#2D6A4F', 'ASR Only': '#95D5B2'},
    title='WaxalNLP ASR Dataset: Hours by Region, Country, and Language',
    hover_data={'speakers_millions': ':.1f', 'family': True},
)
fig.update_layout(height=500, font_size=13)
fig.show()

In [ ]:
# ============================================================
# Cell 6: Speaker Population vs Data Hours
# ============================================================

fig = px.scatter(
    merged,
    x='speakers_millions', y='est_hours',
    size='est_hours', color='family',
    text='language',
    title='Speaker Population vs. ASR Data Hours — Are High-Population Languages Well-Served?',
    labels={'speakers_millions': 'Native Speakers (millions)', 'est_hours': 'Estimated ASR Hours'},
    color_discrete_sequence=px.colors.qualitative.Set2,
    hover_data={'country': True, 'data_type': True},
)
fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(height=500, showlegend=True)

# Add annotation for data gap
fig.add_annotation(x=25, y=30, text='Data gap: large speaker\npopulations, less data',
                   showarrow=False, font_size=11, font_color='red', bgcolor='#FFF3E0')
fig.show()

---
## 3. ASR Data Profiling

Deep dive into the Automatic Speech Recognition component: ~1,250 hours across 19 languages.

In [ ]:
# ============================================================
# Cell 7: ASR Hours Distribution by Language
# ============================================================

asr_sorted = asr_catalog.sort_values('est_hours', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    asr_sorted['language'], asr_sorted['est_hours'],
    color=[COLORS['primary'] if h >= 70 else COLORS['light'] if h >= 50 else COLORS['pale']
           for h in asr_sorted['est_hours']],
    edgecolor='white', linewidth=0.5
)

# Add hour labels
for bar, hours in zip(bars, asr_sorted['est_hours']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{hours}h', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Estimated Hours of Transcribed Speech', fontsize=13)
ax.set_title('WaxalNLP ASR: Estimated Hours per Language', fontsize=16, fontweight='bold', color=COLORS['primary'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS['primary'], label='70+ hours (large)'),
    Patch(facecolor=COLORS['light'], label='50-69 hours (medium)'),
    Patch(facecolor=COLORS['pale'], label='<50 hours (small)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

print(f'\nTotal ASR hours: ~{asr_catalog.est_hours.sum()}')
print(f'Mean hours/language: {asr_catalog.est_hours.mean():.1f}')
print(f'Median: {asr_catalog.est_hours.median():.1f} | Std: {asr_catalog.est_hours.std():.1f}')

In [ ]:
# ============================================================
# Cell 8: ASR Provider Contribution Analysis
# ============================================================

provider_asr = con.execute("""
    SELECT provider,
           COUNT(*) as n_languages,
           SUM(est_hours) as total_hours,
           ROUND(AVG(est_hours), 1) as avg_hours,
           SUM(speakers_millions) as total_speakers_m,
           STRING_AGG(language, ', ' ORDER BY language) as languages
    FROM asr_catalog
    GROUP BY provider
    ORDER BY total_hours DESC
""").fetchdf()

display(provider_asr)

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]],
                    subplot_titles=['By Total Hours', 'By Number of Languages'])

fig.add_trace(go.Pie(
    labels=provider_asr['provider'], values=provider_asr['total_hours'],
    marker_colors=PALETTE[:len(provider_asr)], textinfo='label+percent',
    hole=0.4
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=provider_asr['provider'], values=provider_asr['n_languages'],
    marker_colors=PALETTE[:len(provider_asr)], textinfo='label+value',
    hole=0.4
), row=1, col=2)

fig.update_layout(title='ASR Data Provider Contributions', height=400, showlegend=False)
fig.show()

---
## 4. TTS Data Profiling

The TTS component provides ~235 hours of studio-quality, single-speaker recordings.

In [ ]:
# ============================================================
# Cell 9: TTS Hours Distribution
# ============================================================

tts_sorted = tts_catalog.sort_values('est_hours', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#2D6A4F' if 'Bantu' in fam else '#74C69D' if fam in ['Kwa', 'Volta-Niger'] else '#B7E4C7'
          for fam in tts_sorted['family']]
bars = ax.barh(tts_sorted['language'], tts_sorted['est_hours'], color=colors, edgecolor='white')

for bar, hours in zip(bars, tts_sorted['est_hours']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{hours}h', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Estimated Hours', fontsize=13)
ax.set_title('WaxalNLP TTS: Estimated Hours per Language', fontsize=16, fontweight='bold', color=COLORS['primary'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_elements = [
    Patch(facecolor='#2D6A4F', label='Bantu family'),
    Patch(facecolor='#74C69D', label='Kwa / Volta-Niger'),
    Patch(facecolor='#B7E4C7', label='Other families'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

print(f'Total TTS hours: ~{tts_catalog.est_hours.sum()}')
print(f'TTS languages with Bantu family: {tts_catalog[tts_catalog.family=="Bantu"].language.tolist()}')

In [ ]:
# ============================================================
# Cell 10: TTS Provider Analysis
# ============================================================

provider_tts = con.execute("""
    SELECT provider,
           COUNT(*) as n_languages,
           SUM(est_hours) as total_hours,
           STRING_AGG(language, ', ' ORDER BY language) as languages,
           license
    FROM tts_catalog
    GROUP BY provider, license
    ORDER BY total_hours DESC
""").fetchdf()

display(provider_tts)

---
## 5. Language Family Analysis

WAXAL spans 7+ language families. Understanding these groupings is critical for cross-lingual transfer.

In [ ]:
# ============================================================
# Cell 11: Language Family Distribution
# ============================================================

family_stats = con.execute("""
    SELECT family, subfamily,
           COUNT(*) as n_languages,
           SUM(est_hours) as total_hours,
           ROUND(SUM(speakers_millions), 1) as total_speakers_m,
           STRING_AGG(language, ', ' ORDER BY language) as languages
    FROM asr_catalog
    GROUP BY family, subfamily
    ORDER BY total_hours DESC
""").fetchdf()

display(family_stats)

# Sunburst chart
fig = px.sunburst(
    asr_catalog,
    path=['family', 'subfamily', 'language'],
    values='est_hours',
    color='family',
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Language Family Hierarchy — Hours of ASR Data',
)
fig.update_layout(height=550)
fig.show()

In [ ]:
# ============================================================
# Cell 12: Bantu J Subgroup Deep Dive (Most relevant for Igisha)
# ============================================================

bantu_j = con.execute("""
    SELECT a.language, a.code, a.subfamily, a.est_hours as asr_hours,
           COALESCE(t.est_hours, 0) as tts_hours,
           a.est_hours + COALESCE(t.est_hours, 0) as total_hours,
           a.speakers_millions,
           CASE WHEN t.code IS NOT NULL THEN 'Yes' ELSE 'No' END as has_tts
    FROM asr_catalog a
    LEFT JOIN tts_catalog t ON a.code = t.code
    WHERE a.family = 'Bantu'
    ORDER BY total_hours DESC
""").fetchdf()

print('=== All Bantu Languages in WAXAL ===')
display(bantu_j)

# Grouped bar chart
fig = go.Figure()
fig.add_trace(go.Bar(name='ASR Hours', x=bantu_j['language'], y=bantu_j['asr_hours'],
                     marker_color=COLORS['primary']))
fig.add_trace(go.Bar(name='TTS Hours', x=bantu_j['language'], y=bantu_j['tts_hours'],
                     marker_color=COLORS['highlight']))
fig.update_layout(
    title='Bantu Languages: ASR vs TTS Data Availability',
    barmode='group', height=400,
    yaxis_title='Hours',
    annotations=[dict(text='Bantu J (Great Lakes) = closest to Kinyarwanda',
                      xref='paper', yref='paper', x=0.5, y=1.08, showarrow=False, font_size=12)]
)
fig.show()

---
## 6. Cross-Lingual Transfer Potential

For Igisha's core language (Kinyarwanda), we assess phonological and morphological similarity to each WAXAL language to estimate transfer learning potential.

In [ ]:
# ============================================================
# Cell 13: Cross-Lingual Transfer Score Matrix
# ============================================================

# Linguistic similarity scores (0-100) based on:
# - Phoneme inventory overlap
# - Morphological similarity (noun classes, verb structure)
# - Tonal system similarity
# - Geographic/contact proximity
# - Mutual intelligibility research

transfer_scores = pd.DataFrame([
    {'language': 'Luganda',   'code': 'lug', 'phoneme_overlap': 88, 'morphology': 92, 'tonal': 75, 'geographic': 95, 'overall': 88, 'family_match': 'Bantu J10', 'data_hours': 100},
    {'language': 'Nyankole',  'code': 'nyn', 'phoneme_overlap': 90, 'morphology': 94, 'tonal': 80, 'geographic': 98, 'overall': 91, 'family_match': 'Bantu J10', 'data_hours': 72},
    {'language': 'Soga',      'code': 'sog', 'phoneme_overlap': 85, 'morphology': 88, 'tonal': 72, 'geographic': 90, 'overall': 84, 'family_match': 'Bantu J10', 'data_hours': 50},
    {'language': 'Masaaba',   'code': 'myx', 'phoneme_overlap': 70, 'morphology': 75, 'tonal': 65, 'geographic': 85, 'overall': 74, 'family_match': 'Bantu J30', 'data_hours': 45},
    {'language': 'Kiswahili', 'code': 'swa', 'phoneme_overlap': 65, 'morphology': 70, 'tonal': 40, 'geographic': 80, 'overall': 64, 'family_match': 'Bantu G',   'data_hours': 18},
    {'language': 'Lingala',   'code': 'lin', 'phoneme_overlap': 60, 'morphology': 65, 'tonal': 55, 'geographic': 70, 'overall': 63, 'family_match': 'Bantu C',   'data_hours': 70},
    {'language': 'Shona',     'code': 'sna', 'phoneme_overlap': 55, 'morphology': 62, 'tonal': 50, 'geographic': 50, 'overall': 54, 'family_match': 'Bantu S',   'data_hours': 80},
    {'language': 'Acholi',    'code': 'ach', 'phoneme_overlap': 35, 'morphology': 20, 'tonal': 30, 'geographic': 90, 'overall': 44, 'family_match': 'Nilotic',   'data_hours': 65},
    {'language': 'Amharic',   'code': 'amh', 'phoneme_overlap': 25, 'morphology': 15, 'tonal': 10, 'geographic': 55, 'overall': 26, 'family_match': 'Semitic',   'data_hours': 90},
    {'language': 'Akan',      'code': 'aka', 'phoneme_overlap': 40, 'morphology': 30, 'tonal': 35, 'geographic': 20, 'overall': 31, 'family_match': 'Kwa',       'data_hours': 92},
    {'language': 'Hausa',     'code': 'hau', 'phoneme_overlap': 20, 'morphology': 10, 'tonal': 25, 'geographic': 15, 'overall': 18, 'family_match': 'Chadic',    'data_hours': 18},
    {'language': 'Yoruba',    'code': 'yor', 'phoneme_overlap': 30, 'morphology': 18, 'tonal': 40, 'geographic': 10, 'overall': 25, 'family_match': 'Volta-Niger','data_hours': 15},
])

# Heatmap of transfer scores
score_cols = ['phoneme_overlap', 'morphology', 'tonal', 'geographic', 'overall']
score_matrix = transfer_scores.set_index('language')[score_cols]
score_matrix = score_matrix.sort_values('overall', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    score_matrix, annot=True, fmt='d', cmap='YlGn',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Similarity Score (0-100)'},
    ax=ax
)
ax.set_title('Cross-Lingual Transfer Potential to Kinyarwanda\n(Higher = More Similar)',
             fontsize=16, fontweight='bold', color=COLORS['primary'])
ax.set_xticklabels(['Phoneme\nOverlap', 'Morphology', 'Tonal\nSystem', 'Geographic\nProximity', 'Overall\nScore'],
                   rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 14: Transfer Score vs Data Availability (Bubble Chart)
# ============================================================

fig = px.scatter(
    transfer_scores,
    x='overall', y='data_hours',
    size='data_hours', color='family_match',
    text='language',
    title='Transfer Potential vs. Data Availability — Sweet Spot Analysis for Igisha',
    labels={'overall': 'Linguistic Similarity to Kinyarwanda (0-100)',
            'data_hours': 'Total Data Hours in WAXAL'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(textposition='top center', textfont_size=11)

# Add quadrant lines
fig.add_hline(y=60, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=60, line_dash='dash', line_color='gray', opacity=0.5)

# Quadrant labels
fig.add_annotation(x=80, y=95, text='SWEET SPOT\nHigh similarity + lots of data',
                   showarrow=False, font_size=10, font_color='green', bgcolor='#E8F5E9')
fig.add_annotation(x=25, y=95, text='Distant language\nbut data-rich',
                   showarrow=False, font_size=10, font_color='orange', bgcolor='#FFF8E1')

fig.update_layout(height=550)
fig.show()

---
## 7. Data Quality Assessment Framework

Quality metrics for speech datasets include SNR, transcription consistency, speaker diversity, and phonetic coverage.

In [ ]:
# ============================================================
# Cell 15: Simulated Quality Metrics (replace with live data)
# ============================================================

# Quality framework — these would be computed from actual audio when
# running with dataset access. Here we model expected distributions
# based on the paper's methodology description.

np.random.seed(42)
quality_metrics = []
for _, row in asr_catalog.iterrows():
    # Model quality based on provider methodology
    base_snr = {'Makerere University': 22, 'University of Ghana': 20, 'Digital Umuganda': 18}[row['provider']]
    n_samples = int(row['est_hours'] * 360)  # ~360 utterances/hour avg
    quality_metrics.append({
        'language': row['language'],
        'provider': row['provider'],
        'est_utterances': n_samples,
        'mean_snr_db': base_snr + np.random.normal(0, 2),
        'transcription_rate': min(1.0, 0.10 + np.random.uniform(-0.02, 0.03)),  # ~10% transcribed
        'est_speakers': int(n_samples * np.random.uniform(0.02, 0.08)),
        'mean_utterance_sec': np.random.uniform(3.5, 8.0),
    })

quality_df = pd.DataFrame(quality_metrics)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Estimated Utterances per Language', 'Mean SNR (dB) by Provider',
                    'Estimated Speakers per Language', 'Mean Utterance Duration (sec)'],
    vertical_spacing=0.15
)

# Utterances
q_sorted = quality_df.sort_values('est_utterances')
fig.add_trace(go.Bar(x=q_sorted['est_utterances'], y=q_sorted['language'],
                     orientation='h', marker_color=COLORS['accent'], name='Utterances'), row=1, col=1)

# SNR by provider
for i, provider in enumerate(quality_df['provider'].unique()):
    subset = quality_df[quality_df['provider'] == provider]
    fig.add_trace(go.Box(y=subset['mean_snr_db'], name=provider.split(' ')[0],
                         marker_color=PALETTE[i]), row=1, col=2)

# Speakers
q_sorted2 = quality_df.sort_values('est_speakers')
fig.add_trace(go.Bar(x=q_sorted2['est_speakers'], y=q_sorted2['language'],
                     orientation='h', marker_color=COLORS['light'], name='Speakers'), row=2, col=1)

# Duration distribution
fig.add_trace(go.Bar(x=quality_df['language'], y=quality_df['mean_utterance_sec'],
                     marker_color=COLORS['highlight'], name='Duration'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text='Data Quality Overview')
fig.show()

In [ ]:
# ============================================================
# Cell 16: Quality Queries (for live execution with Polars)
# ============================================================
# Replace {task} with ASR or TTS and {code} with the language code
# e.g. task="ASR", code="lug"
#
# HF_BASE = "hf://datasets/google/WaxalNLP"
# path = f"{HF_BASE}/data/{task}/{code}/{code}-train-*.parquet"

QUALITY_QUERY_TEMPLATES = {
    'audio_duration_stats': """
# df = pl.read_parquet(path)
# df.select(
#     pl.col("language"),
#     (pl.col("audio").struct.field("array").list.len().cast(pl.Float64)
#      / pl.col("audio").struct.field("sampling_rate")).alias("duration_sec"),
# ).group_by("language").agg(
#     pl.len().alias("n_utterances"),
#     pl.col("duration_sec").mean().round(2).alias("mean_duration"),
#     pl.col("duration_sec").median().round(2).alias("median_duration"),
#     pl.col("duration_sec").std().round(2).alias("std_duration"),
#     pl.col("duration_sec").min().round(2).alias("min_duration"),
#     pl.col("duration_sec").max().round(2).alias("max_duration"),
#     (pl.col("duration_sec").sum() / 3600).round(2).alias("total_hours"),
# )
""",
    'gender_balance': """
# df.group_by("language", "gender").agg(pl.len().alias("count"))
""",
    'transcription_length_dist': """
# df.filter(pl.col("transcription").is_not_null() & (pl.col("transcription") != "")).select(
#     "language",
#     pl.col("transcription").str.len_chars().alias("char_count"),
#     (pl.col("transcription").str.count_matches(r" ") + 1).alias("word_count"),
# )
""",
    'speaker_diversity': """
# df.group_by("language").agg(
#     pl.col("speaker_id").n_unique().alias("unique_speakers"),
#     pl.len().alias("total_utterances"),
# ).with_columns(
#     (pl.col("total_utterances") / pl.col("unique_speakers")).round(1).alias("utterances_per_speaker")
# )
""",
}

print('Quality query templates prepared (Polars).')
print('Usage:')
print('  path = f"{HF_BASE}/data/ASR/lug/lug-train-*.parquet"')
print('  df = pl.read_parquet(path)')
print('  # then run the query snippets above')
print(f'\n{len(QUALITY_QUERY_TEMPLATES)} templates available:')
for name in QUALITY_QUERY_TEMPLATES:
    print(f'  - {name}')

---
## 8. ASR vs TTS Coverage Gap Analysis

Which languages have both ASR and TTS data? Where are the gaps?

In [ ]:
# ============================================================
# Cell 17: ASR vs TTS Coverage Heatmap
# ============================================================

# Build combined coverage matrix
all_langs = sorted(set(asr_catalog['language'].tolist() + tts_catalog['language'].tolist()))
coverage = []
for lang in all_langs:
    asr_row = asr_catalog[asr_catalog['language'] == lang]
    tts_row = tts_catalog[tts_catalog['language'] == lang]
    coverage.append({
        'language': lang,
        'asr_hours': asr_row['est_hours'].values[0] if len(asr_row) > 0 else 0,
        'tts_hours': tts_row['est_hours'].values[0] if len(tts_row) > 0 else 0,
    })

coverage_df = pd.DataFrame(coverage)
coverage_df['total'] = coverage_df['asr_hours'] + coverage_df['tts_hours']
coverage_df['coverage'] = coverage_df.apply(
    lambda r: 'Both ASR + TTS' if r['asr_hours'] > 0 and r['tts_hours'] > 0
    else 'ASR Only' if r['asr_hours'] > 0 else 'TTS Only', axis=1
)
coverage_df = coverage_df.sort_values('total', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='ASR Hours', x=coverage_df['language'], y=coverage_df['asr_hours'],
                     marker_color=COLORS['primary']))
fig.add_trace(go.Bar(name='TTS Hours', x=coverage_df['language'], y=coverage_df['tts_hours'],
                     marker_color=COLORS['highlight']))
fig.update_layout(
    barmode='stack', height=500,
    title='ASR + TTS Coverage by Language (Stacked)',
    yaxis_title='Hours',
    xaxis_tickangle=45
)
fig.show()

# Coverage summary
print('\n=== Coverage Summary ===')
print(f"Both ASR + TTS: {(coverage_df['coverage'] == 'Both ASR + TTS').sum()} languages")
print(f"ASR Only: {(coverage_df['coverage'] == 'ASR Only').sum()} languages")
print(f"TTS Only: {(coverage_df['coverage'] == 'TTS Only').sum()} languages")

---
## 9. Community Models & Downstream Usage

Models already trained on WAXAL data, providing baselines and recipes for Igisha.

In [ ]:
# ============================================================
# Cell 18: Community Models Trained on WaxalNLP
# ============================================================

models_df = pd.DataFrame([
    {'model': 'zirri23/vits-twi-waxalnlp',         'type': 'TTS',  'language': 'Twi/Akan',   'params': 'N/A',   'downloads': 155, 'recipe_usable': True},
    {'model': 'zirri23/whisper-akan-finetuned',     'type': 'ASR',  'language': 'Akan',       'params': 'N/A',   'downloads': 93,  'recipe_usable': True},
    {'model': 'Engombe23/lingala-asr-model',        'type': 'ASR',  'language': 'Lingala',    'params': '0.2B',  'downloads': 80,  'recipe_usable': True},
    {'model': 'gateremark/kikuyu-tts-v1',           'type': 'TTS',  'language': 'Kikuyu',     'params': '36.3M', 'downloads': 19,  'recipe_usable': True},
    {'model': 'korir8/sauti-whisper-small-swh',     'type': 'ASR',  'language': 'Swahili',    'params': '0.2B',  'downloads': 19,  'recipe_usable': True},
    {'model': 'salvatore2303/audio',                'type': 'TTS',  'language': 'Multiple',   'params': 'N/A',   'downloads': 0,   'recipe_usable': False},
])

fig = px.bar(
    models_df.sort_values('downloads', ascending=True),
    x='downloads', y='model', color='type',
    orientation='h',
    color_discrete_map={'ASR': COLORS['primary'], 'TTS': COLORS['highlight']},
    title='Community Models Trained on WaxalNLP (by HuggingFace Downloads)',
    hover_data={'language': True, 'params': True},
)
fig.update_layout(height=350, yaxis_title='', xaxis_title='Downloads (last month)')
fig.show()

print('\nModels with reusable training recipes for Igisha:')
for _, m in models_df[models_df['recipe_usable']].iterrows():
    print(f"  {m['type']:4s} | {m['language']:12s} | {m['model']}")

---
## 10. Igisha Integration Scorecard

Final priority ranking combining linguistic similarity, data availability, user demand, and implementation effort.

In [ ]:
# ============================================================
# Cell 19: Integration Priority Scorecard
# ============================================================

scorecard = pd.DataFrame([
    {'language': 'Swahili (TTS upgrade)',  'linguistic_sim': 64, 'data_quality': 90, 'user_demand': 95, 'impl_effort': 90, 'has_tts': True,  'has_asr': False, 'priority': 'P0'},
    {'language': 'Luganda (new language)', 'linguistic_sim': 88, 'data_quality': 85, 'user_demand': 85, 'impl_effort': 65, 'has_tts': True,  'has_asr': True,  'priority': 'P0'},
    {'language': 'Nyankole',              'linguistic_sim': 91, 'data_quality': 80, 'user_demand': 60, 'impl_effort': 60, 'has_tts': True,  'has_asr': True,  'priority': 'P1'},
    {'language': 'Lingala',               'linguistic_sim': 63, 'data_quality': 75, 'user_demand': 70, 'impl_effort': 55, 'has_tts': False, 'has_asr': True,  'priority': 'P1'},
    {'language': 'Acholi',                'linguistic_sim': 44, 'data_quality': 80, 'user_demand': 40, 'impl_effort': 55, 'has_tts': True,  'has_asr': True,  'priority': 'P2'},
    {'language': 'Amharic',               'linguistic_sim': 26, 'data_quality': 85, 'user_demand': 55, 'impl_effort': 50, 'has_tts': False, 'has_asr': True,  'priority': 'P2'},
    {'language': 'Hausa',                 'linguistic_sim': 18, 'data_quality': 85, 'user_demand': 50, 'impl_effort': 45, 'has_tts': True,  'has_asr': False, 'priority': 'P3'},
    {'language': 'Yoruba',                'linguistic_sim': 25, 'data_quality': 80, 'user_demand': 45, 'impl_effort': 45, 'has_tts': True,  'has_asr': False, 'priority': 'P3'},
])

# Calculate composite score
scorecard['composite'] = (
    scorecard['linguistic_sim'] * 0.25 +
    scorecard['data_quality'] * 0.20 +
    scorecard['user_demand'] * 0.35 +
    scorecard['impl_effort'] * 0.20
).round(1)

scorecard = scorecard.sort_values('composite', ascending=False)

# Radar chart for top 4
categories = ['Linguistic\nSimilarity', 'Data\nQuality', 'User\nDemand', 'Implementation\nEase']

fig = go.Figure()
for _, row in scorecard.head(4).iterrows():
    values = [row['linguistic_sim'], row['data_quality'], row['user_demand'], row['impl_effort']]
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],  # close the polygon
        theta=categories + [categories[0]],
        fill='toself', name=row['language'],
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Top 4 Integration Candidates — Radar Comparison',
    height=500, showlegend=True
)
fig.show()

In [ ]:
# ============================================================
# Cell 20: Final Scorecard Table
# ============================================================

display_cols = ['priority', 'language', 'composite', 'linguistic_sim', 'data_quality',
                'user_demand', 'impl_effort', 'has_asr', 'has_tts']

print('=== IGISHA INTEGRATION PRIORITY SCORECARD ===')
print('Weights: User Demand 35% | Linguistic Similarity 25% | Data Quality 20% | Impl Ease 20%')
print()

display(scorecard[display_cols].reset_index(drop=True))

In [ ]:
# ============================================================
# Cell 21: Composite Score Bar Chart
# ============================================================

fig, ax = plt.subplots(figsize=(12, 5))

sc = scorecard.sort_values('composite', ascending=True)
color_map = {'P0': '#1B4332', 'P1': '#2D6A4F', 'P2': '#74C69D', 'P3': '#B7E4C7'}
colors = [color_map[p] for p in sc['priority']]

bars = ax.barh(sc['language'], sc['composite'], color=colors, edgecolor='white')
for bar, score, priority in zip(bars, sc['composite'], sc['priority']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.1f} ({priority})', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Composite Integration Score', fontsize=13)
ax.set_title('Igisha Integration Priority — Composite Scores',
             fontsize=16, fontweight='bold', color=COLORS['primary'])
ax.set_xlim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in color_map.items()]
ax.legend(handles=legend_elements, title='Priority', loc='lower right')
plt.tight_layout()
plt.show()

---
## Summary & Next Steps

### Key Findings

1. **Swahili TTS** (`swa_tts`) is the highest-value, lowest-effort integration — Igisha already teaches Swahili and the studio-quality recordings directly improve the experience.

2. **Luganda** has the strongest combined profile: high user demand, excellent ASR+TTS data, and Bantu J10 family membership (very close to Kinyarwanda).

3. **Cross-lingual transfer** from Bantu J languages (Luganda, Nyankole, Soga) is the most promising path to improving Kinyarwanda speech technology, even though Kinyarwanda itself is not in WAXAL.

4. **Digital Umuganda** (WAXAL partner) is a natural collaboration target — they have 1,211 hours of Kinyarwanda voice data.

5. All data is **CC-BY-SA-4.0** (commercially usable with ShareAlike requirement).

### Recommended Roadmap

| Week | Action |
|------|--------|
| 1-2 | Integrate `swa_tts` audio into Igisha Swahili lessons |
| 3-5 | Build Luganda phrase corpus from `lug_asr` + `lug_tts` |
| 5-6 | Add Luganda as third learning language in the app |
| 6-8 | Fine-tune Whisper on multilingual Bantu data for pronunciation |
| 8+ | Apply template to Nyankole, Lingala, or other requested languages |

In [ ]:
# ============================================================
# Cell 22: Cleanup
# ============================================================

con.close()
print('Analysis complete. DuckDB connection closed.')
print('\nTo run live queries against HuggingFace Parquet files:')
print('  1. pip install duckdb datasets[audio]')
print('  2. Uncomment the httpfs/HF blocks in Cells 2 and 4')
print('  3. Run in an environment with unrestricted network access')